In [ ]:
# Smart Healing: AI-Based Disease Diagnosis System
# Artificial Intelligence (CSC-350) - Spring 2026
# Sukkur IBA University

import numpy as np
import pandas as pd
from collections import deque
import heapq
import random
import math
import warnings
warnings.filterwarnings("ignore")  # Hides unnecessary warning messages from scikit-learn

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    mean_absolute_error, mean_squared_error
)


# ===========================================================
# 1. CREATE PATIENT DATASET
# ===========================================================

def create_dataset():
    np.random.seed(42)  # Ensures the random numbers generated are identical every time you run
    n = 200            # Total number of fake patient profiles to build

    # Simulating standard clinical values using Gaussian/Normal bell-curve distributions
    glucose  = np.clip(np.random.normal(120, 30, n), 60, 200).astype(int)   # Average 120, range 60-200
    bmi      = np.clip(np.random.normal(30, 7, n), 15, 55).round(1)        # Average BMI 30, range 15-55
    age      = np.clip(np.random.normal(35, 12, n), 18, 80).astype(int)    # Adult patients between 18 and 80
    bp       = np.clip(np.random.normal(72, 12, n), 40, 110).astype(int)   # Diastolic Blood Pressure
    insulin  = np.clip(np.random.normal(80, 50, n), 0, 300).astype(int)    # Insulin level tracking
    skin     = np.clip(np.random.normal(25, 10, n), 5, 60).astype(int)     # Triceps skin fold thickness
    preg     = np.clip(np.random.randint(0, 10, n), 0, 10)                 # Number of times pregnant
    pedigree = np.clip(np.random.exponential(0.5, n), 0.05, 2.5).round(3)  # Diabetes family history scale

    # Combining individual feature arrays into a clean structured Pandas DataFrame
    df = pd.DataFrame({
        "Glucose": glucose, "BMI": bmi, "Age": age,
        "BloodPressure": bp, "Insulin": insulin,
        "SkinThickness": skin, "Pregnancies": preg,
        "DiabetesPedigree": pedigree
    })

    # A medical logical rule calculating risk units to determine the target variable
    score = (
        (df["Glucose"] > 125).astype(int) * 2 +  # Over 125 mg/dL is a major risk factor (counts double)
        (df["BMI"] > 30).astype(int) +            # BMI > 30 represents clinical obesity
        (df["Age"] > 40).astype(int) +            # Advancing age scales up diabetes risks
        (df["DiabetesPedigree"] > 0.5).astype(int) # Genetic inheritance indicator
    )

    # Ground-truth assignment: if a profile collects 3 or more risk points, label as Diabetic (1), else Healthy (0)
    df["Disease"] = (score >= 3).astype(int)

    # Requirement fulfilment: physically writes data out so the CSV matches instructor expectations
    df.to_csv("smart_healing_dataset.csv", index=False)

    return df


# ===========================================================
# 2. RATIONAL AGENT (PEAS STRUCTURE)
# ===========================================================

class DiagnosticAgent:
    """
    Implements a classic Intelligent Agent architecture.
    Sensor: Takes raw patient dictionaries.
    Actuator: Emits diagnostic labels and structured assessment dictionaries.
    """
    def __init__(self):
        self.name = "SmartHealingAgent"

    def diagnose(self, patient, model, scaler):
        # Extracts incoming patient dictionary values and shapes them as a 2D sample array for scikit-learn
        data = np.array(list(patient.values())).reshape(1, -1)

        # Scales incoming data using metrics computed from the original training split
        data = scaler.transform(data)

        prediction  = model.predict(data)[0]          # Obtains binary response class (0 or 1)
        probability = model.predict_proba(data)[0][1] # Extracts true confidence score percentage (0.0 to 1.0)

        # Map abstract numbers into friendly descriptive terms
        diagnosis = "Diabetic" if prediction == 1 else "Healthy"
        risk = "HIGH" if probability > 0.7 else "MEDIUM" if probability > 0.4 else "LOW"

        return {
            "Diagnosis"  : diagnosis,
            "Risk Level" : risk,
            "Confidence" : f"{probability * 100:.1f}%"
        }


# ===========================================================
# 3. SEARCH ALGORITHMS (UNINFORMED VS INFORMED)
# ===========================================================

# Graph mapping out clinical pathways: Symptoms connect to temporary conditions leading to diagnoses
SYMPTOM_GRAPH = {
    "start":              [("high_glucose", 1), ("fatigue", 2), ("frequent_urination", 1)],
    "high_glucose":       [("diabetes", 3), ("prediabetes", 2)],
    "fatigue":            [("diabetes", 4), ("anemia", 3)],
    "frequent_urination": [("diabetes", 2), ("uti", 1)],
    "prediabetes":        [("diabetes", 1)],
    "diabetes": [], "anemia": [], "uti": [],
}

# Heuristic values (h): rough intuitive distance approximations remaining to reach target goal 'diabetes'
HEURISTIC = {
    "start": 4, "high_glucose": 2, "fatigue": 3,
    "frequent_urination": 2, "prediabetes": 1,
    "diabetes": 0, "anemia": 5, "uti": 5,
}


def bfs(graph, start, goal):
    """ Breadth-First Search: Explores level by level using a FIFO queue. Finds shortest hop count. """
    queue = deque([[start]])  # Store complete execution paths inside queue
    visited = set()
    while queue:
        path = queue.popleft() # Fetch oldest path from structural front
        node = path[-1]
        if node == goal:
            return path
        if node not in visited:
            visited.add(node)
            for neighbor, _ in graph.get(node, []):
                queue.append(path + [neighbor]) # Appends adjacent steps to end of path queue
    return None


def dfs(graph, start, goal, path=None, visited=None):
    """ Depth-First Search: Explores deep paths completely before backtracking using recursive calls. """
    if path is None:
        path, visited = [start], set()
    if start == goal:
        return path
    visited.add(start)
    for neighbor, _ in graph.get(start, []):
        if neighbor not in visited:
            # Traverses deeper along active branch lines recursively
            result = dfs(graph, neighbor, goal, path + [neighbor], visited)
            if result:
                return result
    return None


def ucs(graph, start, goal):
    """ Uniform Cost Search: Dijkstra variant. Expands paths with least cumulative edge cost (g) using a Min-Heap. """
    heap = [(0, [start])] # Priority Queue tracks tuple structure: (running_cost, path_list)
    visited = set()
    while heap:
        cost, path = heapq.heappop(heap) # Extract path with absolute minimal running cost
        node = path[-1]
        if node == goal:
            return path, cost
        if node not in visited:
            visited.add(node)
            for neighbor, edge_cost in graph.get(node, []):
                heapq.heappush(heap, (cost + edge_cost, path + [neighbor])) # Tracks cumulative path costs
    return None, 0


def astar(graph, heuristic, start, goal):
    """ A* Search: Uses total evaluation function f(n) = g(n) + h(n) to balance path cost and heuristic data. """
    heap = [(heuristic[start], 0, [start])] # Track priorities sorted by: (f_score, actual_cost_g, path)
    visited = set()
    while heap:
        f, g, path = heapq.heappop(heap) # Pop element showcasing minimum total expected cost
        node = path[-1]
        if node == goal:
            return path, g
        if node not in visited:
            visited.add(node)
            for neighbor, cost in graph.get(node, []):
                new_g = g + cost # Actual physical cost calculated so far
                new_f = new_g + heuristic.get(neighbor, 0) # Total estimated path expense function
                heapq.heappush(heap, (new_f, new_g, path + [neighbor]))
    return None, 0


# ===========================================================
# 4. GENETIC ALGORITHM (EVOLUTIONARY OPTIMIZATION)
# ===========================================================

def genetic_algorithm(X_train, y_train, X_test, y_test, n_features=8):
    """ Optimizes feature selection using selection, crossover, and mutation over multiple generations. """

    def fitness(chromosome):
        """ Evaluates utility score based on validation accuracy given active feature indicators (1s). """
        selected = [i for i, b in enumerate(chromosome) if b == 1]
        if not selected:
            return 0.0 # Return absolute zero fitness value if no inputs are active
        model = KNeighborsClassifier(n_neighbors=5)
        model.fit(X_train[:, selected], y_train)
        return accuracy_score(y_test, model.predict(X_test[:, selected]))

    # Instantiating 10 random chromosome options (e.g. [1, 0, 1, 1, 0, 0, 1, 0])
    population = [[random.randint(0, 1) for _ in range(n_features)] for _ in range(10)]
    best, best_score = None, 0.0

    for _ in range(5): # Iterating evolution cycle across 5 full generations
        scored = sorted([(fitness(c), c) for c in population], reverse=True) # Rank population by score
        if scored[0][0] > best_score:
            best_score = scored[0][0]
            best = scored[0][1] # Safekeeping optimal feature vector combination found

        survivors = [c for _, c in scored[:5]] # Survival of the Fittest: preserve top 5 performers
        children = []

        # Crossover phase: Recombine genetic feature sequences between consecutive surviving parent masks
        for i in range(0, len(survivors) - 1, 2):
            point = random.randint(1, n_features - 1) # Choose a pivot junction slicing point
            children.append(survivors[i][:point] + survivors[i+1][point:])
            children.append(survivors[i+1][:point] + survivors[i][point:])

        # Mutation phase: Introduce random bit flips with a 20% probability rate to explore new features
        for child in children:
            for j in range(n_features):
                if random.random() < 0.2:
                    child[j] = 1 - child[j] # Flips bit value (0 becomes 1, 1 becomes 0)

        population = survivors + children # Consolidate surviving members and new off-spring lines

    selected = [i for i, b in enumerate(best) if b == 1]
    return selected, best_score


# ===========================================================
# 5. MINIMAX WITH ALPHA-BETA PRUNING (ADVERSARIAL SEARCH)
# ===========================================================

# Hierarchical representation of clinical testing options. Path evaluations show medical priority scales.
GAME_TREE = {
    "name": "Order Tests?",
    "children": [
        {"name": "Blood Test", "children": [
            {"name": "Glucose Test", "value": 9, "children": []},
            {"name": "HbA1c Test",   "value": 7, "children": []},
        ]},
        {"name": "Skip Tests", "children": [
            {"name": "Use History",  "value": 5, "children": []},
            {"name": "Random Guess", "value": 2, "children": []},
        ]},
    ]
}


def minimax(node, depth, is_max, alpha, beta):
    """ Computes optimal utility strategies while pruning irrelevant tree pathways. """
    if depth == 0 or not node.get("children"):
        return node.get("value", 0) # Return leaf node terminal numeric valuation

    if is_max:
        best = float('-inf')
        for child in node["children"]:
            best = max(best, minimax(child, depth - 1, False, alpha, beta))
            alpha = max(alpha, best) # Update lower boundary search constraints
            if beta <= alpha:
                break # Alpha Cutoff: Prunes branches that the minimizing opponent would never allow
        return best
    else:
        best = float('inf')
        for child in node["children"]:
            best = min(best, minimax(child, depth - 1, True, alpha, beta))
            beta = min(beta, best)  # Update upper boundary search constraints
            if beta <= alpha:
                break # Beta Cutoff: Prunes paths because the maximizing player already has a better option
        return best


# ===========================================================
# 6. SUPERVISED MACHINE LEARNING TRIPLE DEPLOYMENT
# ===========================================================

def train_models(X_train, y_train):
    """ Trains three distinct classification paradigms to evaluate relative system performance. """

    # 1. Instance-Based: Classifies cases checking spatial proximity boundaries
    knn = KNeighborsClassifier(n_neighbors=7)
    knn.fit(X_train, y_train)

    # 2. Structural Logic: Generates consecutive threshold information splitting questions
    dt = DecisionTreeClassifier(max_depth=6, random_state=42)
    dt.fit(X_train, y_train)

    # 3. Probabilistic Linear Model: Maps logarithmic log-odds outcomes to Sigmoid boundaries
    lr = LogisticRegression(max_iter=200, random_state=42)
    lr.fit(X_train, y_train)

    return knn, dt, lr


# ===========================================================
# 7. MULTI-CRITERIA STATISTICAL EVALUATION METRICS
# ===========================================================

def evaluate(model, X_test, y_test, name):
    """ Calculates standard performance metrics beyond simple accuracy ratios. """
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Risk likelihood probabilities for positive target case

    # Extract foundational core quadrants from the evaluation confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Medical Metric Formulas
    acc         = accuracy_score(y_test, y_pred) * 100
    precision   = precision_score(y_test, y_pred, zero_division=0) * 100 # Positive Predictive Value
    sensitivity = recall_score(y_test, y_pred, zero_division=0) * 100    # True Positive Rate (CRITICAL for matching sick patients)
    specificity = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0          # True Negative Rate (Correctly clearing healthy individuals)
    f1          = f1_score(y_test, y_pred, zero_division=0)               # Balanced harmonic mean matching precision and recall
    geo_mean    = math.sqrt((sensitivity / 100) * (specificity / 100))    # Balance score for uneven label classes
    auc         = roc_auc_score(y_test, y_prob)                           # Separability threshold rating performance curves
    mae         = mean_absolute_error(y_test, y_pred)                      # Distance tracking average absolute errors
    mse         = mean_squared_error(y_test, y_pred)                       # Penalizes larger prediction error instances

    # Displaying all computed results inside terminal window outputs
    print(f"\n  Model       : {name}")
    print(f"  Accuracy    : {acc:.1f}%")
    print(f"  Precision   : {precision:.1f}%")
    print(f"  Sensitivity : {sensitivity:.1f}%")
    print(f"  Specificity : {specificity:.1f}%")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  Geo Mean    : {geo_mean:.4f}")
    print(f"  AUC-ROC     : {auc:.4f}")
    print(f"  MAE         : {mae:.4f}")
    print(f"  MSE         : {mse:.4f}")
    print(f"  Confusion Matrix -> TP={tp} FP={fp} FN={fn} TN={tn}")

    return f1 # Return F1-Score to help identify the highest performing model


# ===========================================================
# 8. SYSTEM COMPILATION CONTROL (MAIN RUN METHOD)
# ===========================================================

def main():
    print("\n" + "=" * 55)
    print("   SMART HEALING: AI-BASED DISEASE DIAGNOSIS")
    print("   Artificial Intelligence (CSC-350) - Spring 2026")
    print("   Sukkur IBA University")
    print("=" * 55)

    # ── STEP 1: Dataset Validation & CSV Generation ──
    print("\n--- 1. PATIENT DATASET ---")
    df = create_dataset() # Invokes dataset code which writes file records to local path
    print(f"   Total Patients : {len(df)}")
    print(f"   Healthy        : {(df['Disease'] == 0).sum()}")
    print(f"   Diabetic       : {(df['Disease'] == 1).sum()}")
    print(f"\n   First 5 rows:")
    print(df.head().to_string(index=False))

    # ── STEP 2: PEAS Environment Verification ──
    print("\n--- 2. RATIONAL AGENT (PEAS) ---")
    print("   Performance : Accuracy, Sensitivity, F1-Score")
    print("   Environment : Patient records, lab results")
    print("   Actuators   : Diagnosis, Risk Level, Confidence")
    print("   Sensors     : Glucose, BMI, Age, BloodPressure, etc.")

    # ── STEP 3: Classic Pathfinding Graph Search ──
    print("\n--- 3. SEARCH ALGORITHMS ---")
    goal = "diabetes"
    path = bfs(SYMPTOM_GRAPH, "start", goal)
    print(f"   BFS : {' -> '.join(path)}")
    path = dfs(SYMPTOM_GRAPH, "start", goal)
    print(f"   DFS : {' -> '.join(path)}")
    path, cost = ucs(SYMPTOM_GRAPH, "start", goal)
    print(f"   UCS : {' -> '.join(path)}  (cost={cost})")
    path, cost = astar(SYMPTOM_GRAPH, HEURISTIC, "start", goal)
    print(f"   A* : {' -> '.join(path)}  (cost={cost})")

    # ── STEP 4: Matrix Separation & Partitioning ──
    X = df.drop("Disease", axis=1).values # Isolates measurement data inputs from classification categories
    y = df["Disease"].values              # Isolates truth flags

    # Stratification divides subsets making sure identical target class weight metrics persist across both halves
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # StandardScaler transforms records converting variances to unity and centering mean distributions on zero
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train) # Computes conversion scale criteria using training array parameters
    X_test_sc  = scaler.transform(X_test)      # Transforms test matrix keeping scale settings uniform
    feature_names = list(df.columns[:-1])

    # ── STEP 5: Genetic Feature Subset Extraction ──
    print("\n--- 4. GENETIC ALGORITHM (Feature Selection) ---")
    selected_idx, ga_score = genetic_algorithm(X_train_sc, y_train, X_test_sc, y_test)
    selected = [feature_names[i] for i in selected_idx]
    print(f"   Original : {feature_names}")
    print(f"   Selected : {selected}")
    print(f"   Score    : {ga_score * 100:.2f}%")

    # ── STEP 6: Minimax Strategy Evaluation ──
    print("\n--- 5. MIN-MAX + ALPHA-BETA PRUNING ---")
    best_val = minimax(GAME_TREE, depth=3, is_max=True, alpha=float('-inf'), beta=float('inf'))
    print(f"   Best Score     : {best_val}")
    print(f"   Best Action    : Order Blood Test -> Glucose Test")

    # ── STEP 7: Classifiers Training Execution ──
    print("\n--- 6. MACHINE LEARNING MODELS ---")
    knn, dt, lr = train_models(X_train_sc, y_train)
    print("   Trained: kNN, Decision Tree, Logistic Regression")

    # ── STEP 8: Performance Metrics Cross-Analysis ──
    print("\n--- 7. EVALUATION METRICS ---")
    f1_knn = evaluate(knn, X_test_sc, y_test, "kNN (k=7)")
    f1_dt  = evaluate(dt,  X_test,    y_test, "Decision Tree") # Note: Tree architectures do not require feature scaling transformations
    f1_lr  = evaluate(lr, X_test_sc, y_test, "Logistic Regression")

    best_name = ["kNN", "Decision Tree", "Logistic Regression"][
        [f1_knn, f1_dt, f1_lr].index(max(f1_knn, f1_dt, f1_lr))
    ]
    print(f"\n   Best Model : {best_name}")

    # ── STEP 9: K-Fold Structural Stability Check ──
    print("\n--- 8. K-FOLD CROSS VALIDATION ---")
    kf = KFold(n_splits=5, shuffle=True, random_state=42) # Split data into 5 unique stratified rotation sets
    scores = cross_val_score(LogisticRegression(max_iter=200), X_train_sc, y_train, cv=kf)
    print(f"   Mean Accuracy : {scores.mean() * 100:.2f}%")
    print(f"   Std Dev       : ±{scores.std() * 100:.2f}%") # Indicates if performance varies heavily across variations
    print(f"   Fold Scores   : {[round(s * 100, 1) for s in scores]}")

    # ── STEP 10: Information Gain Attribute Importance ──
    print("\n--- 9. FEATURE IMPORTANCE (Decision Tree) ---")
    ranked = sorted(zip(feature_names, dt.feature_importances_), key=lambda x: x[1], reverse=True)
    for i, (feat, imp) in enumerate(ranked, 1):
        bar = "█" * int(imp * 40) # Generates simple character-based horizontal bar visualization chart
        print(f"   {i}. {feat:<20} {imp:.4f}  {bar}")

    # ── STEP 11: Unsupervised Profile Pattern Structuring ──
    print("\n--- 10. K-MEANS CLUSTERING ---")
    kmeans   = KMeans(n_clusters=2, random_state=42, n_init=10) # Groups records by feature spatial properties alone
    clusters = kmeans.fit_predict(X_train_sc)
    print(f"   Cluster 0 (Low Risk)  : {(clusters == 0).sum()} patients")
    print(f"   Cluster 1 (High Risk) : {(clusters == 1).sum()} patients")

    # ── STEP 12: Independent Inference Diagnostic Evaluation ──
    print("\n--- 11. DIAGNOSE A NEW PATIENT ---")
    agent = DiagnosticAgent()
    new_patient = {
        "Glucose": 155, "BMI": 34.5, "Age": 48,
        "BloodPressure": 80, "Insulin": 120,
        "SkinThickness": 30, "Pregnancies": 3,
        "DiabetesPedigree": 0.8
    }
    print(f"   Input : {new_patient}")
    result = agent.diagnose(new_patient, lr, scaler) # Uses the robust Logistic Regression baseline instance
    for key, val in result.items():
        print(f"   {key:<12}: {val}")

    print("\n" + "=" * 55)
    print("   All sections completed successfully.")
    print("=" * 55 + "\n")


if __name__ == "__main__":
    main()